QCGPT Quantum Spec Tests

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if (root / 'qcgpt').exists():
    sys.path.insert(0, str(root))
elif (root.parent / 'qcgpt').exists():
    sys.path.insert(0, str(root.parent))


In [2]:
import numpy as np
import torch
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity
from qcgpt.data.specs import state_pairs_to_spec_tensor, spec_tensor_to_state_pairs, build_spec_sequence_batch
from qcgpt.simulators.qiskit_sim import circuit_to_qiskit, basis_bits_to_statevector
from qcgpt.models.policy import CircuitPolicy
from qcgpt.gates import VOCAB, BOS_CIRC_ID, EOS_CIRC_ID, PAD_ID
from qcgpt.encoding import tokens_to_circuit
from qcgpt.data.qiskit_utils import random_reference_circuit
from qcgpt.training.rl import rl_step
from qcgpt.training.rollouts import RewardBaseline
from qcgpt.evaluation.metrics import quantum_fidelity_from_spec


ModuleNotFoundError: No module named 'qiskit'

Test 1: Qiskit states → spec_tensor → back

In [ ]:
qc = QuantumCircuit(2)
qc.cx(0,1)
psi_pairs = []
for bits in [(0,0),(1,1)]:
    psi_in = basis_bits_to_statevector(np.array(bits))
    psi_out = psi_in.evolve(qc)
    psi_pairs.append((psi_in, psi_out))
spec = state_pairs_to_spec_tensor(psi_pairs, n_qubits=2)
pairs_back = spec_tensor_to_state_pairs(spec, n_qubits=2)
errs = []
for (psi_in, psi_out), (rin, rout) in zip(psi_pairs, pairs_back):
    errs.append(np.max(np.abs(psi_in.data - rin)))
    errs.append(np.max(np.abs(psi_out.data - rout)))
max_err = float(np.max(errs))
max_err


Test 2: spec_tensor → spec_batch → SpecEncoder

In [ ]:
spec_batch_np, spec_pad_mask_np = build_spec_sequence_batch([spec])
device = torch.device('cpu')
model = CircuitPolicy(vocab_size=len(VOCAB)).to(device)
enc_out = model.encoder(torch.tensor(spec_batch_np, dtype=torch.float32),
                       torch.tensor(spec_pad_mask_np, dtype=torch.bool))
enc_out.shape


Test 3: spec_batch + CircuitPolicy → circuit tokens

In [ ]:
spec_batch = torch.tensor(spec_batch_np, dtype=torch.float32)
spec_pad_mask = torch.tensor(spec_pad_mask_np, dtype=torch.bool)
with torch.no_grad():
    seqs, logp = model.sample_circuit_tokens(spec_batch, spec_pad_mask, BOS_CIRC_ID, EOS_CIRC_ID, max_len=16)
seq = [t for t in seqs[0].tolist() if t != PAD_ID]
circ = tokens_to_circuit(seq)
len(seq), circ.gates[:3]


Test 4: candidate circuit → Qiskit fidelity vs target

In [ ]:
ref_circ = random_reference_circuit()
spec_ref = state_pairs_to_spec_tensor([(basis_bits_to_statevector(np.array([0,0])),
                                        basis_bits_to_statevector(np.array([0,0])).evolve(circuit_to_qiskit(ref_circ)))], 2)
fid_ref = quantum_fidelity_from_spec(spec_ref, ref_circ)
fid_cand = quantum_fidelity_from_spec(spec_ref, circ)
fid_ref, fid_cand


Test 5: RL step + gradients

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
mean_R, loss_val = rl_step(model, opt, device, batch_size=2, max_len=16, lambda_len=0.1, max_gates_ref=6, baseline=RewardBaseline(), use_qiskit_fidelity=True)
loss_val
